In [ ]:
from pathlib import Path
from functools import lru_cache, cached_property

In [ ]:
from sj_utils.file.json import load_json

In [ ]:
# OUTPUT = "/workspaces/dev/output/libri/clean/rt_whisper_each_with_se47_4.json"
OUTPUT_1 = "/workspaces/dev/test/performance_test/esic/output/20250730/validation_rt-whisper-each-p.json"
OUTPUT_2 = "/workspaces/dev/test/performance_test/esic/output/20250730/validation_rt-whisper-each.json"

In [ ]:
CP = "correct_percent"
SP = "substitution_percent"
DP = "deletion_percent"
IP = "insertion_percent"
WER = "wer_percent"
SEP = "sentence_error_percent"
METRIC = [CP, SP, DP, IP, WER, SEP]

@lru_cache(maxsize=128)
def get_data(path: str):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    _, js = load_json(path)
    for k in js:
        if "processed_time" in js[k]:
            del js[k]["processed_time"]
        if "transcribe_time" in js[k]:
            del js[k]["transcribe_time"]

    return js

def metric_view(max_name: str, max_val: float, min_name: str, min_val: float):
    return f"\t{'max:':<8} {max_name:<20} ({max_val:.4f})  -  {'min:':<8} {min_name:<20} ({min_val:.4f})"

class Metric:
    def __init__(self, name:str, data:dict):
        for key, value in data.items():
            for m in METRIC:
                if m not in value:
                    raise ValueError(f"Metric {m} not found in data for {key}.")
        self.name = name
        self.data = data

    @cached_property
    def sorted_cp(self):
        return sorted(
            [{"name": k, "metric": v} for k, v in self.data.items()],
            key=lambda x: x["metric"][CP]
        )

    @cached_property
    def sorted_sp(self):
        return sorted(
            [{"name": k, "metric": v} for k, v in self.data.items()],
            key=lambda x: x["metric"][SP], reverse=True
        )

    @cached_property
    def sorted_dp(self):
        return sorted(
            [{"name": k, "metric": v} for k, v in self.data.items()],
            key=lambda x: x["metric"][DP], reverse=True
        )

    @cached_property
    def sorted_ip(self):
        return sorted(
            [{"name": k, "metric": v} for k, v in self.data.items()],
            key=lambda x: x["metric"][IP], reverse=True
        )

    @cached_property
    def sorted_wer(self):
        return sorted(
            [{"name": k, "metric": v} for k, v in self.data.items()],
            key=lambda x: x["metric"][WER], reverse=True
        )

    def show_all_metric(self, top_k:int = 1):
        scp = self.sorted_cp[:top_k], reversed(self.sorted_cp[-top_k:])
        ssp = self.sorted_sp[:top_k], reversed(self.sorted_sp[-top_k:])
        sdp = self.sorted_dp[:top_k], reversed(self.sorted_dp[-top_k:])
        sip = self.sorted_ip[:top_k], reversed(self.sorted_ip[-top_k:])
        swer = self.sorted_wer[:top_k], reversed(self.sorted_wer[-top_k:])

        print(f"{self.name} - Correct Percent:")
        for t, b in zip(*scp):
            print(metric_view(t['name'], t['metric'][CP], b['name'], b['metric'][CP]))
        print(f"{self.name} - Substitution Percent:")
        for t, b in zip(*ssp):
            print(metric_view(t['name'], t['metric'][SP], b['name'], b['metric'][SP]))
        print(f"{self.name} - Deletion Percent:")
        for t, b in zip(*sdp):
            print(metric_view(t['name'], t['metric'][DP], b['name'], b['metric'][DP]))
        print(f"{self.name} - Insertion Percent:")
        for t, b in zip(*sip):
            print(metric_view(t['name'], t['metric'][IP], b['name'], b['metric'][IP]))
        print(f"{self.name} - WER Percent:")
        for t, b in zip(*swer):
            print(metric_view(t['name'], t['metric'][WER], b['name'], b['metric'][WER]))

    def analyze_frequency(self, top_k:int = 1):
        scp = self.sorted_cp[:top_k]
        ssp = self.sorted_sp[:top_k]
        sdp = self.sorted_dp[:top_k]
        sip = self.sorted_ip[:top_k]
        swer = self.sorted_wer[:top_k]

        frequency = {}
        for t in scp + ssp + sdp + sip + swer:
            name = t['name']
            if name not in frequency:
                frequency[name] = 0
            frequency[name] += 1

        sorted_frequency = sorted(frequency.items(), key=lambda x: x[1], reverse=True)
        print(f"Frequency of top {top_k} metrics:")
        for name, count in sorted_frequency:
            print(f"{name}: {count} times")

    def __sub__(self, other):
        if not isinstance(other, Metric):
            raise TypeError("Subtraction is only supported between Metric instances.")

        result_data = {}
        for key in self.data:
            if not key in other.data:
                raise KeyError(f"Key {key} not found in both Metric instances for subtraction.")
            result_data[key] = {
                "num_sentences": self.data[key]["num_sentences"],
                "num_words": self.data[key]["num_words"],
                "correct_percent": self.data[key][CP] - other.data[key][CP],
                "substitution_percent": self.data[key][SP] - other.data[key][SP],
                "deletion_percent": self.data[key][DP] - other.data[key][DP],
                "insertion_percent": self.data[key][IP] - other.data[key][IP],
                "wer_percent": self.data[key][WER] - other.data[key][WER],
                "sentence_error_percent": self.data[key][SEP] - other.data[key][SEP],
            }

        return Metric(f"{self.name} - {other.name}", result_data)

In [ ]:
output_1 = get_data(OUTPUT_1)
output_2 = get_data(OUTPUT_2)

In [ ]:
metric_p = Metric("rt_whisper-p", output_1["rt_whisper"])
metric = Metric("rt_whisper", output_2["rt_whisper"])

In [ ]:
metric_p.show_all_metric(top_k = 4)

In [ ]:
metric.show_all_metric(top_k = 4)

In [ ]:
# metric.analyze_frequency(top_k = 10)
# metric_p.analyze_frequency(top_k = 10)

In [ ]:
m_diff = metric - metric_p

In [ ]:
m_diff.show_all_metric(top_k = 20)